# Tier-1 subway/systemwide supply from NTD monthly VRM [WP-A1, Path 2 fallback] — rev.2

Builds a monthly **service-supply** series from the FTA NTD Monthly Module's **Vehicle Revenue
Miles (VRM)** for subway, buses, LIRR, and Metro-North. **No GTFS needed**; reaches 2019, so it
delivers the subway's structural-gap anchor and a systemwide supply-vs-demand picture in one step.

**rev.2:** handles the real workbook layout — a single long-format **`Data`** sheet (one row per
agency/mode/month with `UPT/VRM/VRH/VOMS` columns and a `Date` column), not a wide per-metric
sheet. `load_ntd_monthly(path, metric="VRM")` auto-detects the sheet and layout.

**Confirmed on the 2026-05 release (offline run):** subway VRM ~29.5M car-mi/mo (≈354M/yr, matches
NYCT's known annual figure); SIR is a **separate** NTD reporter (Staten Island Rapid Transit
Operating Authority, 20099), so NYCT Heavy Rail is subway-only. **Headline result:** subway
service-per-rider vs 2019 = **1.31** — service held near 2019 while demand sits at ~76%.


## 0 · Config & load (long 'Data' sheet, VRM)

In [ ]:
import pathlib
import pandas as pd
import ntd_vrm as n

RAW = pathlib.Path("data/raw")
if not RAW.exists(): RAW = pathlib.Path("..")/"data"/"raw"
RAW.mkdir(parents=True, exist_ok=True)
NTD_XLSX = RAW / "ntd_monthly.xlsx"
assert NTD_XLSX.exists(), f"Place the NTD monthly workbook at {NTD_XLSX}."

long = n.load_ntd_monthly(str(NTD_XLSX), metric="VRM")   # auto: 'Data' sheet, long layout
print("VRM rows:", len(long), "| months:",
      long.period_start.min().date(), "->", long.period_start.max().date())


## 1 · Discover the MTA agency/mode series (confirm before mapping)

In [ ]:
mta = long[long.agency.astype(str).str.contains(
    "New York City Transit|MTA Bus|Long Island Rail Road|Metro-North|Staten Island", case=False, na=False)]
print(mta.groupby(["ntd_id","agency","mode","tos"])["value"].agg(n="count", latest_max="max")
        .reset_index().sort_values(["agency","mode"]).to_string(index=False))


## 2 · Map to paper services and aggregate
Confirmed mapping (real agency names / NTD IDs): subway = NYCT HR (20008); buses = NYCT MB (20008)
+ MTA Bus MB (20188); LIRR = 20100 CR; Metro-North = 20078 CR. SIR (20099 HR) already has its own
scheduled series (k3aj) — left out here; add only as an operated cross-check. NYCT bus CB/RB
sub-modes (small, from ~2012) are omitted for a clean, continuously-reported core.

In [ ]:
RULES = [
    dict(service="Subways",     mode_class="subway", mode="HR", agency_regex="New York City Transit"),
    dict(service="Buses",       mode_class="bus",    mode=["MB","CB","RB"], agency_regex="New York City Transit|MTA Bus"),
    dict(service="LIRR",        mode_class="rail",   mode="CR", agency_regex="Long Island Rail Road"),
    dict(service="Metro-North", mode_class="rail",   mode="CR", agency_regex="Metro-North"),
]
series = {r["service"]: n.aggregate_service(long, r["service"], r["mode"], r["agency_regex"]) for r in RULES}
for r in RULES:
    s = series[r["service"]]
    print(f"{r['service']:12s} {s.index.min().date()}..{s.index.max().date()} n={len(s)}  "
          f"2019={s[s.index.year==2019].mean():,.0f}  latest12={s.tail(12).mean():,.0f}")


## 3 · Validate (continuity + level cross-checks, not rescaling)
Index cancels any offset, so no rescale. Check continuity, flag the provisional recent tail, and
sanity-check the level (subway ~29.5M car-mi/mo ≈ 354M/yr matches NYCT annual).

In [ ]:
for name, s in series.items():
    gaps = pd.date_range(s.index.min(), s.index.max(), freq="MS").difference(s.index)
    print(f"{name:12s} gaps={len(gaps)}  annualized latest={s.tail(12).sum():,.0f} {'car' if name!='Buses' else 'veh'}-mi/yr")
print("\nNOTE: monthly module lags ~2 months and estimates the most recent months -> provisional tail.")


## 4 · Emit rows + subway service-per-rider vs 2019 (the anchor prize)

In [ ]:
rows = pd.concat([n.to_feed_rows(series[r["service"]], r["service"], r["mode_class"], r["mode"])
                  for r in RULES], ignore_index=True)
rows.to_csv(RAW / "ntd_vrm_monthly_preview.csv", index=False)
print("wrote", RAW / "ntd_vrm_monthly_preview.csv", "rows:", len(rows))

feed = pathlib.Path("data/feeds/mta_chart_feed_multigrain.csv")
if not feed.exists(): feed = pathlib.Path("..")/"data"/"feeds"/"mta_chart_feed_multigrain.csv"
if feed.exists():
    mg = pd.read_csv(feed, parse_dates=["period_start"]); mm = mg[mg.grain=="monthly"]
    def dem(s): return mm[(mm.service==s)&(mm.measure_type=="estimated_ridership")&(mm.evidence=="observed")].set_index("period_start")["value"].sort_index()
    def idx(x,yr): return x/x[x.index.year==yr].mean()*100
    d = idx(dem("Subways"),2019).rolling(3,min_periods=1).mean(); s = idx(series["Subways"],2019).rolling(3,min_periods=1).mean()
    c = d.index.intersection(s.index); spr = s.loc[c]/d.loc[c]
    print("\nSubway service-per-rider vs 2019:")
    for yr in [2019,2020,2022,2024,2026]:
        print(f"  {yr}: supply={s[s.index.year==yr].mean():5.1f} demand={d[d.index.year==yr].mean():5.1f} spr={spr[spr.index.year==yr].mean():.2f}")
    print(f"  latest 3-mo service-per-rider = {spr.tail(3).mean():.2f}  (service held ahead of 2019-relative demand)")


## 5 · Next steps & caveats
1. Append `ntd_vrm_monthly_preview.csv` to the multigrain feed (variation-stat cols NaN, per the
   derived-series convention). Add `derived_ntd_vrm` to the provenance docs; add supply constants.
2. The systemwide figure `fig_supply_demand_systemwide.png` (subway vs 2019 + all modes vs 2022) is
   the systemwide capstone. Subway vs 2019 = 1.31 (service held ahead); vs 2022 the rail modes lag
   (MNR 0.69, LIRR 0.84) as demand outpaced VRM since the 2022 trough; buses ~1.08.
3. **Measure caveat to state:** NTD VRM = *operated car/vehicle revenue-miles* (a capacity measure),
   distinct from SIR *scheduled train-trips* and LIRR *operated train-trips*; each on its own axis,
   indexed, never pooled. LIRR VRM grew ~+25% at GCM vs +43% in operated train-trips — GCM added
   trips but shorter average runs, so keep the occupancy trip-count as LIRR's primary (matches the
   published +41%) and use VRM for the subway (its only source) and the systemwide overlay.
4. Add `check_ntd_vrm_is_derived` QA. Note the ~2-month lag / provisional recent months.
